In [2]:
from pathlib import Path
desktop = Path.home() / "Desktop"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate

# Construction of the Hamiltonian
def ising_hamiltonian(n, J, h):
    paulis = []
    coeffs = []
    for i in range(n):
        for j in range(i+1, n):
            label = ["I"] * n
            label[i] = "Z"
            label[j] = "Z"
            paulis.append("".join(label))
            coeffs.append(J)
    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)
    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

nqubits = 3
Hamiltonian = ising_hamiltonian(nqubits, 1, 0.5)

# Accurate Quantum Circuit and Quantum State Simulation
def build_ansatz(params, ops_list):
    qc = QuantumCircuit(nqubits)
    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)
        if len(op_str) == 1:
            for q in range(nqubits):
                qc.append(PauliEvolutionGate(op, time=param), [q])
        else:
            for q in range(nqubits - 1):
                qc.append(PauliEvolutionGate(op, time=param), [q, q+1])
    return qc

def get_energy(params, ops_list, H):
    qc = build_ansatz(params, ops_list)
    sv = Statevector(qc)
    return np.real(sv.expectation_value(H))

def get_V(params, ops_list, H, epsilon=1e-5):
    V = np.zeros(len(params))
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        V[i] = -(get_energy(p_plus, ops_list, H) - get_energy(p_minus, ops_list, H)) / (2 * epsilon)
    return V

def get_M(params, ops_list, epsilon=1e-5):
    sv_center = Statevector(build_ansatz(params, ops_list)).data
    derivatives = []
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        d_sv = (Statevector(build_ansatz(p_plus, ops_list)).data - Statevector(build_ansatz(p_minus, ops_list)).data) / (2 * epsilon)
        derivatives.append(d_sv)
        
    num_p = len(params)
    M = np.zeros((num_p, num_p))
    for i in range(num_p):
        for j in range(num_p):
            term1 = np.vdot(derivatives[i], derivatives[j])
            term2 = np.vdot(derivatives[i], sv_center) * np.vdot(sv_center, derivatives[j])
            M[i, j] = 2 * np.real(term1 + term2)
    return M

def get_residual(params, ops_list, H, M, V):
    e_val = get_energy(params, ops_list, H)
    H_sq = H.compose(H).simplify()
    e_sq_val = get_energy(params, ops_list, H_sq)
    var_H = e_sq_val - e_val**2
    
    M_reg = M + 0.01 * np.eye(len(params))
    M_inv = np.linalg.pinv(M_reg)
    residual = 2 * var_H - V.T @ M_inv @ V
    return residual

# Dynamic Growth Training Loop
def dynamic_vite_training(H, exact_energy, max_iters=60, learning_rate=0.05, threshold=0.05):
    CANDIDATE_OPS = ["XX", "YY", "ZZ", "XY", "YX", "XZ"]
    Ops = ["X", "Y", "Z"]
    current_params = np.random.uniform(-0.01, 0.01, len(Ops))
    
    energy_history = []
    depth_history = []
    residual_history = []
    
    print("Dynamic Variational Imaginary-Time Evolution Training")
    
    for iteration in range(max_iters):
        M = get_M(current_params, Ops)
        V = get_V(current_params, Ops, H)
        
        M_reg = M + 0.01 * np.eye(len(Ops))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_energy = get_energy(current_params, Ops, H)
        residual = get_residual(current_params, Ops, H, M, V)
        
        energy_history.append(current_energy)
        depth_history.append(len(Ops))
        residual_history.append(residual)
        
        print(f"Iteration {iteration+1:02d} Number of Parameters {len(Ops):02d} Energy {current_energy:.4f} Residual {residual:.4f}")
        
        if residual > threshold and len(CANDIDATE_OPS) > 0 and (current_energy - exact_energy) > 0.05:
            print("Residual Exceeds the Threshold and Energy Has Not Converged")
            new_op = CANDIDATE_OPS.pop(0)
            Ops.append(new_op)
            current_params = np.append(current_params, 0.0)
            print(f"Entangling Gates Have Been Dynamically Added {new_op}")
        else:
            current_params = current_params + learning_rate * d_theta
            
    return current_params, energy_history, depth_history, residual_history

# Execution and Plotting
exact_ground_energy = -2.14575
final_params, energies, depths, residuals = dynamic_vite_training(Hamiltonian, exact_ground_energy)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 10))

ax1.plot(energies, marker='o', label='Energy', color='blue')
ax1.axhline(y=exact_ground_energy, color='gray', linestyle='--', label='Exact Ground State (-2.14575)')
ax1.set_ylabel('Energy (Hartree)')
ax1.set_title('Dynamic Ansatz VITE Optimization')
ax1.legend()

ax2.plot(depths, marker='s', label='Parameter Count', color='green', drawstyle='steps-post')
ax2.set_ylabel('Parameter Count')
ax2.legend()

ax3.plot(residuals, marker='^', label='McLachlan Residual', color='red')
ax3.set_ylabel('Residual')
ax3.set_xlabel('Iterations')
ax3.legend()

plt.tight_layout()
fig.savefig(desktop/'vite_simulation.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Fixed-Depth Deep Ansatz
FIXED_OPS = ["X", "Y", "Z", "XX", "YY", "ZZ", "XY", "YX", "XZ"]

def static_gd_training(H, exact_energy, max_iters=60, learning_rate=0.05):
    current_params = np.random.uniform(-0.5, 0.5, len(FIXED_OPS))
    energy_history = []
    
    print("Initialization of Static Gradient Descent (GD) Training")
    for iteration in range(max_iters):
        # The V Vector Is Intrinsically the Negative Gradient Direction.
        V = get_V(current_params, FIXED_OPS, H)
        current_params = current_params + learning_rate * V
        
        current_energy = get_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
        
    return energy_history

def static_vite_training(H, exact_energy, max_iters=60, learning_rate=0.05):
    current_params = np.random.uniform(-0.5, 0.5, len(FIXED_OPS))
    energy_history = []
    
    print("Initialization of Static Variational Imaginary-Time Evolution (VITE) Training")
    for iteration in range(max_iters):
        M = get_M(current_params, FIXED_OPS)
        V = get_V(current_params, FIXED_OPS, H)
        
        M_reg = M + 0.01 * np.eye(len(FIXED_OPS))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_params = current_params + learning_rate * d_theta
        
        current_energy = get_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
        
    return energy_history

# Conducting Control Experiments
gd_energies = static_gd_training(Hamiltonian, exact_ground_energy)
static_vite_energies = static_vite_training(Hamiltonian, exact_ground_energy)

# Plotting Comprehensive Comparison Charts
plt.figure(figsize=(10, 6))

plt.plot(gd_energies, marker='x', linestyle=':', color='orange', label='Baseline A: Static GD (9 params)')
plt.plot(static_vite_energies, marker='s', linestyle='--', color='green', label='Baseline B: Static VITE (9 params)')
plt.plot(energies, marker='o', linestyle='-', color='blue', label='Dynamic VITE (Growth from 3 params)')

plt.axhline(y=exact_ground_energy, color='gray', linestyle='-', label='Exact Ground State (-2.14575)')

plt.ylabel('Energy (Hartree)')
plt.xlabel('Iterations')
plt.title('Performance Comparison: Dynamic Growth vs Static Ansätze')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(desktop/'vite_simulation_1.svg', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate

# ==========================================
# 1. System Size and Physical Parameter Settings (Transverse-Field Ising Model)
# ==========================================
nqubits = 6  # You can change
J_coupling = 1.0
h_field = 1.0

def ising_hamiltonian(n, J, h):
    paulis = []
    coeffs = []
    # ZZ Interaction
    for i in range(n):
        for j in range(i+1, n):
            label = ["I"] * n
            label[i] = "Z"
            label[j] = "Z"
            paulis.append("".join(label))
            coeffs.append(J)
    # X-Direction Transverse Magnetic Field
    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)
    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

Hamiltonian = ising_hamiltonian(nqubits, J_coupling, h_field)

# Automatic Diagonalization for Exact Ground-State Energy Calculation of Arbitrary N-Qubit Systems
H_matrix = Hamiltonian.to_matrix()
eigenvalues = eigh(H_matrix, eigvals_only=True)
exact_ground_energy = eigenvalues[0]

print(f"System Size: {nqubits} Qubits")
print(f"Exact Ground-State Energy at the Phase Transition Point: {exact_ground_energy:.4f} Hartree\n")

# ==========================================
# 2. Exact State-Vector Simulation Backend Functions (Eliminating Shot Noise)
# ==========================================
def build_ansatz(params, ops_list):
    """Construction of a Parameter-Sharing Hardware-Efficient Ansatz"""
    qc = QuantumCircuit(nqubits)
    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)
        if len(op_str) == 1:
            for q in range(nqubits):
                qc.append(PauliEvolutionGate(op, time=param), [q])
        else:
            for q in range(nqubits - 1):
                qc.append(PauliEvolutionGate(op, time=param), [q, q+1])
    return qc

def get_energy(params, ops_list, H):
    qc = build_ansatz(params, ops_list)
    sv = Statevector(qc)
    return np.real(sv.expectation_value(H))

def get_V(params, ops_list, H, epsilon=1e-5):
    """Obtaining the V Vector (Negative Gradient) via Numerical Differentiation"""
    V = np.zeros(len(params))
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        V[i] = -(get_energy(p_plus, ops_list, H) - get_energy(p_minus, ops_list, H)) / (2 * epsilon)
    return V

def get_M(params, ops_list, epsilon=1e-5):
    """Obtaining the M Matrix (A Matrix)"""
    sv_center = Statevector(build_ansatz(params, ops_list)).data
    derivatives = []
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        d_sv = (Statevector(build_ansatz(p_plus, ops_list)).data - Statevector(build_ansatz(p_minus, ops_list)).data) / (2 * epsilon)
        derivatives.append(d_sv)
        
    num_p = len(params)
    M = np.zeros((num_p, num_p))
    for i in range(num_p):
        for j in range(num_p):
            term1 = np.vdot(derivatives[i], derivatives[j])
            term2 = np.vdot(derivatives[i], sv_center) * np.vdot(sv_center, derivatives[j])
            M[i, j] = 2 * np.real(term1 + term2)
    return M

def get_residual(params, ops_list, H, M, V):
    """Computing the Squared McLachlan Residual"""
    e_val = get_energy(params, ops_list, H)
    H_sq = (H @ H).simplify()
    e_sq_val = get_energy(params, ops_list, H_sq)
    var_H = e_sq_val - e_val**2
    
    # Adding a Regularization Term to Ensure Stable Matrix Inversion
    M_reg = M + 0.01 * np.eye(len(params))
    M_inv = np.linalg.pinv(M_reg)
    residual = 2 * var_H - V.T @ M_inv @ V
    return residual

# ==========================================
# 3. Training Algorithm Modules (Baseline and Dynamic Approaches)
# ==========================================
FIXED_OPS = ["X", "Y", "Z", "XX", "YY", "ZZ", "XY", "YX", "XZ"]

def static_gd_training(H, max_iters=50, learning_rate=0.05):
    """Baseline A: Static Gradient Descent (GD)"""
    current_params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    energy_history = []
    
    print("Starting Baseline A: Static Gradient Descent (GD) (9 Params)")
    for iteration in range(max_iters):
        V = get_V(current_params, FIXED_OPS, H)
        current_params = current_params + learning_rate * V
        current_energy = get_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
    return energy_history

def static_vite_training(H, max_iters=50, learning_rate=0.05):
    """Baseline B: Static VITE"""
    current_params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    energy_history = []
    
    print("Executing Baseline B: Static VITE (9 Params)")
    for iteration in range(max_iters):
        M = get_M(current_params, FIXED_OPS)
        V = get_V(current_params, FIXED_OPS, H)
        
        M_reg = M + 0.01 * np.eye(len(FIXED_OPS))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_params = current_params + learning_rate * d_theta
        current_energy = get_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
    return energy_history

def dynamic_vite_training(H, exact_energy, max_iters=50, learning_rate=0.05, threshold=0.05):
    """Experimental Group: Dynamic VITE"""
    CANDIDATE_OPS = ["XX", "YY", "ZZ", "XY", "YX", "XZ", "ZX", "YZ", "ZY"]
    Ops = ["X", "Y", "Z"]
    current_params = np.random.uniform(-0.01, 0.01, len(Ops))
    
    energy_history = []
    
    print("Dynamic VITE Training Started (From 3 Params)")
    for iteration in range(max_iters):
        M = get_M(current_params, Ops)
        V = get_V(current_params, Ops, H)
        
        M_reg = M + 0.01 * np.eye(len(Ops))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_energy = get_energy(current_params, Ops, H)
        residual = get_residual(current_params, Ops, H, M, V)
        energy_history.append(current_energy)
        
        # Residual-Triggered Mechanism
        if residual > threshold and len(CANDIDATE_OPS) > 0 and (current_energy - exact_energy) > 0.05:
            new_op = CANDIDATE_OPS.pop(0)
            Ops.append(new_op)
            current_params = np.append(current_params, 0.0)
            print(f"  [Iter {iteration+1:02d}] Residual Too High ({residual:.4f})，Quantum Gates Dynamically Added: {new_op}")
        else:
            current_params = current_params + learning_rate * d_theta
            
    print(f"Final Parameter Count of Dynamic VITE: {len(Ops)}\n")
    return energy_history

# ==========================================
# 4. Plotting Comprehensive Comparison Charts
# ==========================================
iters = 50
gd_energies = static_gd_training(Hamiltonian, max_iters=iters)
svite_energies = static_vite_training(Hamiltonian, max_iters=iters)
dvite_energies = dynamic_vite_training(Hamiltonian, exact_ground_energy, max_iters=iters)

plt.figure(figsize=(10, 6))

plt.plot(gd_energies, marker='x', linestyle=':', color='orange', label='Baseline A: Static GD (9 params)')
plt.plot(svite_energies, marker='s', linestyle='--', color='green', label='Baseline B: Static VITE (9 params)')
plt.plot(dvite_energies, marker='o', linestyle='-', color='blue', label='Dynamic VITE (Growth from 3 params)')

plt.axhline(y=exact_ground_energy, color='gray', linestyle='-', label=f'Exact Ground State ({exact_ground_energy:.4f})')

plt.ylabel('Energy (Hartree)')
plt.xlabel('Iterations')
plt.title(f'Performance Comparison at Critical Point ({nqubits} Qubits)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(desktop/'vite_simulation_2.svg', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, DensityMatrix
from qiskit.circuit.library import PauliEvolutionGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# ==========================================
# 1. System and Noise Model Configuration
# ==========================================
nqubits = 6  
J_coupling = 1.0
h_field = 1.0

# Establishing a Depolarizing Noise Model
noise_model = NoiseModel()
# Setting Error Rates Consistent with the Current State of NISQ Devices
error_1q = depolarizing_error(0.001, 1)  # 0.1% Single-qubit errors
error_2q = depolarizing_error(0.01, 2)   # 1.0% Two-qubit errors
noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz'])
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

# Using a density matrix simulator to obtain exact noisy expectation values, eliminating shot-noise fluctuations.
noisy_sim = AerSimulator(method='density_matrix', noise_model=noise_model)

def ising_hamiltonian(n, J, h):
    paulis, coeffs = [], []
    for i in range(n):
        for j in range(i+1, n):
            label = ["I"] * n
            label[i], label[j] = "Z", "Z"
            paulis.append("".join(label))
            coeffs.append(J)
    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)
    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

Hamiltonian = ising_hamiltonian(nqubits, J_coupling, h_field)
exact_ground_energy = eigh(Hamiltonian.to_matrix(), eigvals_only=True)[0]

print(f"System Size: {nqubits} Qubits")
print(f"Exact Ground-State Energy: {exact_ground_energy:.4f} Hartree\n")

# ==========================================
# 2. Noisy Measurement Backend Function
# ==========================================
def build_ansatz(params, ops_list):
    qc = QuantumCircuit(nqubits)
    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)
        if len(op_str) == 1:
            for q in range(nqubits):
                qc.append(PauliEvolutionGate(op, time=param), [q])
        else:
            for q in range(nqubits - 1):
                qc.append(PauliEvolutionGate(op, time=param), [q, q+1])
    return qc

def get_noisy_energy(params, ops_list, H):
    qc = build_ansatz(params, ops_list)
    # Decomposing PauliEvolution into Elementary Gates to Apply the Noise Model
    qc_t = transpile(qc, basis_gates=['rx', 'ry', 'rz', 'cx'], optimization_level=0)
    qc_t.save_density_matrix()
    
    result = noisy_sim.run(qc_t).result()
    rho = result.data()['density_matrix']
    # Track Noisy Energy Calculations
    return np.real(np.trace(rho.data @ H.to_matrix()))

def get_noisy_V(params, ops_list, H, epsilon=1e-4):
    V = np.zeros(len(params))
    for i in range(len(params)):
        p_plus, p_minus = np.copy(params), np.copy(params)
        p_plus[i] += epsilon
        p_minus[i] -= epsilon
        V[i] = -(get_noisy_energy(p_plus, ops_list, H) - get_noisy_energy(p_minus, ops_list, H)) / (2 * epsilon)
    return V

# To ensure the positive semidefiniteness of the M matrix and the stability of matrix inversion, 
# the ideal metric tensor is retained, while realistic noise is introduced into the driving term 
# V and the residual.
def get_M_ideal(params, ops_list, epsilon=1e-4):
    sv_center = DensityMatrix(build_ansatz(params, ops_list)).data
    derivatives = []
    for i in range(len(params)):
        p_plus, p_minus = np.copy(params), np.copy(params)
        p_plus[i] += epsilon
        p_minus[i] -= epsilon
        rho_plus = DensityMatrix(build_ansatz(p_plus, ops_list)).data
        rho_minus = DensityMatrix(build_ansatz(p_minus, ops_list)).data
        derivatives.append((rho_plus - rho_minus) / (2 * epsilon))
        
    num_p = len(params)
    M = np.zeros((num_p, num_p))
    for i in range(num_p):
        for j in range(num_p):
            term1 = np.trace(derivatives[i] @ derivatives[j])
            term2 = np.trace(derivatives[i] @ sv_center) * np.trace(sv_center @ derivatives[j])
            M[i, j] = 2 * np.real(term1 + term2)
    return M

def get_noisy_residual(params, ops_list, H, M, V):
    e_val = get_noisy_energy(params, ops_list, H)
    H_sq = (H @ H).simplify()
    e_sq_val = get_noisy_energy(params, ops_list, H_sq)
    # Noise-induced variance inflation
    var_H = e_sq_val - e_val**2 
    
    M_reg = M + 0.05 * np.eye(len(params))  # Increasing the Regularization Coefficient to Mitigate Noise
    M_inv = np.linalg.pinv(M_reg)
    residual = 2 * var_H - V.T @ M_inv @ V
    return residual

# ==========================================
# 3. Stress-Testing the Training Loop
# ==========================================
def dynamic_vite_noisy_stress_test(H, exact_energy, max_iters=50, learning_rate=0.03, threshold=0.2):
    CANDIDATE_OPS = ["XX", "YY", "ZZ", "XY", "YX", "XZ"]
    Ops = ["X", "Y", "Z"]
    current_params = np.random.uniform(-0.01, 0.01, len(Ops))
    
    energy_history, depth_history, residual_history = [], [], []
    
    print(" Depolarizing Noise Active")
    for iteration in range(max_iters):
        M = get_M_ideal(current_params, Ops)
        V = get_noisy_V(current_params, Ops, H)
        
        M_reg = M + 0.05 * np.eye(len(Ops))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_energy = get_noisy_energy(current_params, Ops, H)
        residual = get_noisy_residual(current_params, Ops, H, M, V)
        
        energy_history.append(current_energy)
        depth_history.append(len(Ops))
        residual_history.append(residual)
        
        print(f"Iter: {iteration+1:02d} | Parameter: {len(Ops):02d} | Energy: {current_energy:.4f} | Residual: {residual:.4f}")
        
        # Triggering Criterion Under Noisy Conditions: Large Residuals Below the Noise Floor
        if residual > threshold and len(CANDIDATE_OPS) > 0 and (current_energy - exact_energy) > 0.15:
            new_op = CANDIDATE_OPS.pop(0)
            Ops.append(new_op)
            current_params = np.append(current_params, 0.0)
            print(f"  >>> Triggering Dynamic Growth Under Noise Perturbations: Adding {new_op}")
        else:
            current_params = current_params + learning_rate * d_theta
            
    return energy_history, depth_history, residual_history

noisy_energies, noisy_depths, noisy_residuals = dynamic_vite_noisy_stress_test(Hamiltonian, exact_ground_energy)

# ==========================================
# 4. Plotting and Exporting Results
# ==========================================
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 10))

ax1.plot(noisy_energies, marker='o', color='purple', label='Noisy Dynamic VITE Energy')
ax1.axhline(y=exact_ground_energy, color='gray', linestyle='--', label=f'Exact Ground State ({exact_ground_energy:.4f})')
ax1.set_ylabel('Energy (Hartree)')
ax1.set_title('Dynamic Ansatz VITE under Depolarizing Noise')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(noisy_depths, marker='s', color='green', drawstyle='steps-post')
ax2.set_ylabel('Parameter Count')
ax2.grid(True, linestyle='--', alpha=0.6)

ax3.plot(noisy_residuals, marker='^', color='red')
ax3.set_ylabel('McLachlan Residual')
ax3.set_xlabel('Iterations')
ax3.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
fig.savefig(desktop/'vite_simulation_4.svg', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, DensityMatrix
from qiskit.circuit.library import PauliEvolutionGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

nqubits = 10  
J_coupling = 1.0
h_field = 1.0

noise_model = NoiseModel()
error_1q = depolarizing_error(0.001, 1)
error_2q = depolarizing_error(0.01, 2)
noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz'])
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

noisy_sim = AerSimulator(method='density_matrix', noise_model=noise_model)

def ising_hamiltonian(n, J, h):
    paulis, coeffs = [], []
    for i in range(n):
        for j in range(i+1, n):
            label = ["I"] * n
            label[i], label[j] = "Z", "Z"
            paulis.append("".join(label))
            coeffs.append(J)
    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)
    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

Hamiltonian = ising_hamiltonian(nqubits, J_coupling, h_field)
exact_ground_energy = eigh(Hamiltonian.to_matrix(), eigvals_only=True)[0]

print(f"System Size: {nqubits} Qubits")
print(f"Exact Ground Energy: {exact_ground_energy:.4f} Hartree\n")

def build_ansatz(params, ops_list):
    qc = QuantumCircuit(nqubits)
    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)
        if len(op_str) == 1:
            for q in range(nqubits):
                qc.append(PauliEvolutionGate(op, time=param), [q])
        else:
            for q in range(nqubits - 1):
                qc.append(PauliEvolutionGate(op, time=param), [q, q+1])
    return qc

def get_noisy_energy(params, ops_list, H):
    qc = build_ansatz(params, ops_list)
    qc_t = transpile(qc, basis_gates=['rx', 'ry', 'rz', 'cx'], optimization_level=0)
    qc_t.save_density_matrix()
    result = noisy_sim.run(qc_t).result()
    rho = result.data()['density_matrix']
    return np.real(np.trace(rho.data @ H.to_matrix()))

def get_noisy_V(params, ops_list, H, epsilon=1e-4):
    V = np.zeros(len(params))
    for i in range(len(params)):
        p_plus, p_minus = np.copy(params), np.copy(params)
        p_plus[i] += epsilon
        p_minus[i] -= epsilon
        V[i] = -(get_noisy_energy(p_plus, ops_list, H) - get_noisy_energy(p_minus, ops_list, H)) / (2 * epsilon)
    return V

def get_M_ideal(params, ops_list, epsilon=1e-4):
    sv_center = DensityMatrix(build_ansatz(params, ops_list)).data
    derivatives = []
    for i in range(len(params)):
        p_plus, p_minus = np.copy(params), np.copy(params)
        p_plus[i] += epsilon
        p_minus[i] -= epsilon
        rho_plus = DensityMatrix(build_ansatz(p_plus, ops_list)).data
        rho_minus = DensityMatrix(build_ansatz(p_minus, ops_list)).data
        derivatives.append((rho_plus - rho_minus) / (2 * epsilon))
        
    num_p = len(params)
    M = np.zeros((num_p, num_p))
    for i in range(num_p):
        for j in range(num_p):
            term1 = np.trace(derivatives[i] @ derivatives[j])
            term2 = np.trace(derivatives[i] @ sv_center) * np.trace(sv_center @ derivatives[j])
            M[i, j] = 2 * np.real(term1 + term2)
    return M

def get_noisy_residual(params, ops_list, H, M, V):
    e_val = get_noisy_energy(params, ops_list, H)
    H_sq = (H @ H).simplify()
    e_sq_val = get_noisy_energy(params, ops_list, H_sq)
    var_H = e_sq_val - e_val**2 
    M_reg = M + 0.05 * np.eye(len(params))
    M_inv = np.linalg.pinv(M_reg)
    residual = 2 * var_H - V.T @ M_inv @ V
    return residual

FIXED_OPS = ["X", "Y", "Z", "XX", "YY", "ZZ", "XY", "YX", "XZ"]

def static_gd_noisy_training(H, max_iters=50, learning_rate=0.03):
    current_params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    energy_history = []
    print("Run Noisy Static GD")
    for iteration in range(max_iters):
        V = get_noisy_V(current_params, FIXED_OPS, H)
        current_params = current_params + learning_rate * V
        current_energy = get_noisy_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
    return energy_history

def static_vite_noisy_training(H, max_iters=50, learning_rate=0.03):
    current_params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    energy_history = []
    print("Run Noisy Static VITE")
    for iteration in range(max_iters):
        M = get_M_ideal(current_params, FIXED_OPS)
        V = get_noisy_V(current_params, FIXED_OPS, H)
        M_reg = M + 0.05 * np.eye(len(FIXED_OPS))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        current_params = current_params + learning_rate * d_theta
        current_energy = get_noisy_energy(current_params, FIXED_OPS, H)
        energy_history.append(current_energy)
    return energy_history

def dynamic_vite_noisy_training(H, exact_energy, max_iters=50, learning_rate=0.03, threshold=0.15):
    """Dynamic-Growth VITE (Corrected Noisy Growth Logic and Tracking)"""
    CANDIDATE_OPS = ["XX", "YY", "ZZ", "XY", "YX", "XZ"]
    Ops = ["X", "Y", "Z"]
    current_params = np.random.uniform(-0.01, 0.01, len(Ops))
    
    energy_history = []
    
    print("Dynamic VITE Training Started (From 3 Params)")
    for iteration in range(max_iters):
        M = get_M_ideal(current_params, Ops)
        V = get_noisy_V(current_params, Ops, H)
        
        M_reg = M + 0.05 * np.eye(len(Ops))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_energy = get_noisy_energy(current_params, Ops, H)
        residual = get_noisy_residual(current_params, Ops, H, M, V)
        energy_history.append(current_energy)
        
        print(f"  [Iter {iteration+1:02d}] Parameter: {len(Ops):02d} | Energy: {current_energy:.4f} | Residual: {residual:.4f}")
        
        # Correction: Update Parameters First to Ensure Progress at Every Iteration
        current_params = current_params + learning_rate * d_theta
        
        # Correction: Perform Residual-Based Trigger Evaluation (Lower the Threshold to Accommodate Noise)
        if residual > threshold and len(CANDIDATE_OPS) > 0 and (current_energy - exact_energy) > 0.15:
            new_op = CANDIDATE_OPS.pop(0)
            Ops.append(new_op)
            # Initialize New Parameters to Zero to Preserve the Current Evolutionary State
            current_params = np.append(current_params, 0.0)
            print(f"    >>> Dynamic Growth Triggered: Add {new_op}")
            
    print(f"Final Number of Parameters in Dynamic VITE: {len(Ops)}\n")
    return energy_history

iters = 60
gd_noisy_energies = static_gd_noisy_training(Hamiltonian, max_iters=iters)
svite_noisy_energies = static_vite_noisy_training(Hamiltonian, max_iters=iters)
dvite_noisy_energies = dynamic_vite_noisy_training(Hamiltonian, exact_ground_energy, max_iters=iters)

plt.figure(figsize=(10, 6))
plt.plot(gd_noisy_energies, marker='x', linestyle=':', color='orange', label='Noisy Static GD (9 params)')
plt.plot(svite_noisy_energies, marker='s', linestyle='--', color='green', label='Noisy Static VITE (9 params)')
plt.plot(dvite_noisy_energies, marker='o', linestyle='-', color='blue', label='Noisy Dynamic VITE (Growth)')
plt.axhline(y=exact_ground_energy, color='gray', linestyle='-', label=f'Exact Ground State ({exact_ground_energy:.4f})')
plt.ylabel('Energy (Hartree)')
plt.xlabel('Iterations')
plt.title(f'Noisy Performance Comparison ({nqubits} Qubits)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig(desktop/'vite_simulation_5.svg', bbox_inches='tight')
plt.show()